# Paired statistics, grounding diagnostics, and figures

Only manifest-validated complete runs are reported. F1/EM use the official HotpotQA answer normalization in percentage points; equivalence is tested against the downloaded upstream scorer. Confidence intervals resample questions, preserving treatment pairing. A sign randomization test supplements the interval. E1 has one prespecified primary interaction; other tests are exploratory with within-experiment Holm correction. Across-experiment comparisons remain exploratory.

Required outputs: condition/evidence regime/graph F1 and EM, supporting-fact recall, actual payload tokens, full provider input/output usage, logical calls; paired-effect figures and separate payload/inference frontiers. Unknown provider usage remains missing. Retain all helped, harmed and unchanged cases. Gold-supported delivery is a diagnostic, not proof of the mechanism or source entailment.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'protocol.py').exists() else Path.cwd() / 'WEBExtension'
if not (ROOT / 'protocol.py').exists():
    raise FileNotFoundError('Launch Jupyter from the project root or WEBExtension folder')
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))
from protocol import load_official, write_json, file_hash
from experiments import load_config, preview, freeze, run
CFG = load_config(ROOT / 'config.json')
EXECUTE = False  # Set True only after configuring your real backend and maximum call budget.
print('Notebook root:', ROOT)
print('Inference enabled:', EXECUTE)


Notebook root: /home/saurab/Documents/projectRLMAIN/WEBExtension
Inference enabled: False


In [2]:
from reporting import report, physical_usage
from IPython.display import display, Image
RUN_DIR, _=freeze(CFG)
for experiment in ['E1','E2','E3','E4']:
    status=RUN_DIR/(experiment+'_status.json')
    if not status.exists() or json.loads(status.read_text()).get('status')!='COMPLETE':
        print(experiment+': NOT COMPLETED; no results generated')
        continue
    table,stats=report(RUN_DIR,experiment)
    display(table);display(pd.DataFrame(stats['effects'])) if 'pd' in globals() else print(stats['effects'])
    display(Image(filename=str(RUN_DIR/'reports'/experiment/'paired_effects.png')))
print('Physical usage:',physical_usage(RUN_DIR))

E1: NOT COMPLETED; no results generated
E2: NOT COMPLETED; no results generated
E3: NOT COMPLETED; no results generated
E4: NOT COMPLETED; no results generated
Physical usage: {'status': 'NOT_RUN', 'attempted_calls': 0, 'input_tokens': None, 'output_tokens': None}


## Cost interpretation and reporting boundaries

Logical costs include each condition's shared initial calls. Physical billing sums successful real requests once from the call ledger; cache reads are not new completions. Error calls can be billable with unknown usage. Payload counts exclude JSON prompt framing; provider input/output counts include actual inference usage when reported. Do not infer monetary costs without the deployment's actual billing rates and usage records. Measured latency belongs to actual calls, not cached responses. Exported JSON includes exact raw record references for evidence-trace examples.

In [3]:
# Verify raw response files and freeze a shareable checksum index.
checks={str(p.relative_to(ROOT)):file_hash(p) for p in RUN_DIR.rglob('*') if p.is_file() and p.name!='export_checksums.json'}
write_json(RUN_DIR/'export_checksums.json',checks)
print('Checksum entries:',len(checks))

Checksum entries: 19
